# Graha semantic segmentation inference
This notebook runs the toy semantic-segmentation workflow on Pipeline-generated Lunar WAC datacubes. It uses Graha/Lunar-FM for inference and supports the seven WAC VIS+UV bands followed by any selected Kaguya static bands.

The input directory is expected to contain matching WAC and `Static` GeoTIFFs. Pairing and static-band filtering are delegated to `lfm.toy_model.sem_seg.data_cube_inference`, while Graha task construction follows the full-model semantic workflow.

# Setup

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
logging.getLogger('rasterio._env').setLevel(logging.ERROR)

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

In [ ]:
# Run from lfm/notebooks/toy_model, or adjust this for your HPC checkout.
repo_root = Path.cwd().parents[1]
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
if not (repo_root / 'lfm').exists():
    raise FileNotFoundError('Cannot find lfm/. Run this notebook from the repository notebooks directory.')
sys.path.insert(0, str(repo_root))

from lfm.all_models.sem_seg import build_graha_notebook_configs
from lfm.all_models.all_tasks.utils.common import _extract_logits
from lfm.toy_model.sem_seg.data_cube_inference import (
    create_binary_colormap,
    get_datacube_data,
    sliding_window_inference,
)
from lfm.full_model.sem_seg import semantic_graha_components
print('Successfully imported LFM and Graha modules')

# User configuration
- `GRAHA_PRETRAIN_DIR`: to the Graha pretraining experiment directory
    - The pretraining directory must contain `checkpoints/checkpoint_weights_final.pt`, `full_config.yaml`, and `modality_info.yaml`. 
- `GRAHA_LIGHTNING_CHECKPOINT` to the fine-tuned semantic-segmentation Lightning checkpoint.
`BAND_FILTER` is applied after the seven WAC bands and filtered static bands are concatenated. Leave it as `None` to keep all available bands, or provide the exact combined zero-based indices.

In [ ]:
INPUT_ROOT_DIR = Path('/explore/nobackup/projects/lfm/model_inputs/inference/WAC_Processed_AOI')
GRAHA_PRETRAIN_DIR = Path('/explore/nobackup/projects/lfm/gabby/Lunar-FM/experiments/lunarfm_base_dual_full_nas_no_nans_256_256_lr1e-4_wd0.05')
GRAHA_LIGHTNING_CHECKPOINT = Path('/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/graha/model.ckpt')
OUTPUT_DIR = Path('./outputs/inference')
BAND_FILTER = None
MODEL_NATIVE_SIZE = 256
TILE_OVERLAP = 0.25
THRESHOLD = 0.5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Build the Graha inference task
The explicit `wac` backend tells Graha to accept the concatenated WAC+static tensor as one image modality. This is the supported path when the static subset is not exactly Graha's native 63-band static modality.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
notebook_configs = build_graha_notebook_configs(
    output_dir=OUTPUT_DIR,
    data_root=INPUT_ROOT_DIR,
    graha_base_output_dir=OUTPUT_DIR,
    graha_pretrain_dir=GRAHA_PRETRAIN_DIR,
    graha_lightning_checkpoint=GRAHA_LIGHTNING_CHECKPOINT,
    dataset_modality='wac_static',
    graha_input_modality_mode='single',
    graha_backend_modalities=['wac'],
    semantic_label_source='semantic',
    validate_paths=False,
)
config = notebook_configs.experiment_config
graha_config = notebook_configs.graha_config
deps = notebook_configs.dependencies
print(f'Graha backend modalities: {graha_config.graha_backend_modalities}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# Read one datacube to determine the post-filter channel count.
images_raw, file_pairs = get_datacube_data(
    INPUT_ROOT_DIR, band_filter=BAND_FILTER, verbose=True, verify_bands=True
)
if images_raw.size == 0:
    raise ValueError('No valid WAC/Static datacube pairs were found.')
n_channels = int(images_raw.shape[1])
print(f'Loaded {len(file_pairs)} datacube pair(s) with {n_channels} input channels')

In [ ]:
task_cls = semantic_graha_components.make_downstream_shape_segmentation_task_class(
    deps['LunarShapeSegmentationTask']
)
sample_batch = {'image': torch.zeros(1, n_channels, MODEL_NATIVE_SIZE, MODEL_NATIVE_SIZE)}
graha_task = semantic_graha_components.create_task(graha_config, task_cls, sample_batch).to(device)
semantic_graha_components.inspect_backbone(graha_task)
semantic_graha_components.load_lightning_checkpoint_state(
    graha_task, GRAHA_LIGHTNING_CHECKPOINT, 'Graha'
)
graha_task.eval()

class GrahaLogitModel(nn.Module):
    def __init__(self, task):
        super().__init__()
        self.task = task

    def forward(self, image):
        return _extract_logits(self.task(image))

model = GrahaLogitModel(graha_task).to(device).eval()
print('Successfully loaded Graha semantic checkpoint')

# Inference
The legacy datacube helper min-max scales each channel. Graha's custom WAC modality is configured with a `[-1, 1]` input range, so the scaled tensors are remapped before sliding-window inference.

In [ ]:
images_hwc = np.transpose(images_raw, (0, 2, 3, 1)).astype(np.float32)
images_graha = np.empty_like(images_hwc)
for index, image in enumerate(images_hwc):
    band_min = np.nanmin(image, axis=(0, 1), keepdims=True)
    band_max = np.nanmax(image, axis=(0, 1), keepdims=True)
    denominator = np.where(band_max > band_min, band_max - band_min, 1.0)
    images_graha[index] = 2.0 * ((image - band_min) / denominator) - 1.0

preds, probabilities = sliding_window_inference(
    images_graha, model=model, device=device, target_size=MODEL_NATIVE_SIZE,
    threshold=THRESHOLD, n_channels=n_channels, overlap=TILE_OVERLAP, window='hann'
)

fig, axes = plt.subplots(2, len(file_pairs), figsize=(6 * len(file_pairs), 10), squeeze=False)
for index, ((wac_file, static_file), image, pred) in enumerate(zip(file_pairs, images_graha, preds)):
    axes[0, index].imshow(image[:, :, 0], cmap='gray')
    axes[0, index].set_title(Path(wac_file).stem)
    axes[1, index].imshow(create_binary_colormap(pred))
    axes[1, index].set_title(f'Graha prediction ({int(pred.sum()):,} positive pixels)')
    for row in axes[:, index]:
        row.axis('off')
fig.suptitle(f'Graha inference: {n_channels} WAC + static channels', y=1.0)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'graha_inference_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved visualization to {OUTPUT_DIR / "graha_inference_viz.png"}')